In [1]:
import pandas as pd
import os
from features_reindex import get_feature, read_data, read_data_timecut
from model_diffusion import evaluate_disease
import pickle
import sys
import multiprocessing as mp
from sklearn.preprocessing import MinMaxScaler


root = '/itf-fi-ml/shared/users/ziyuzh/svm'

# time_spilt = True
# feature = 'ppi_'+str(time)

time_spilt = True
test_bug = True
# test_bug = False

if test_bug:
    # feature_list = ['uniport_ppi_2019','uniport_bio','uniport_seq','uniport_esm']
    # feature_list = ['ppi_2019','bioconcept']
    # feature_list = ['uniport_ppi_2017','ppi_2017_dw_80','uniport_exp','uniport_seq','uniport_esm']
    # feature_list = ['uniport_ppi_2017','ppi_2017_dw_80','uniport_exp','uniport_seq']
    # feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm']
    feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm','diffusion_2019']
    # feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','diffusion_2019']
    # feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm']


    # dga = 'opentarget'
    dga = 'disgenet'

    out_path = os.path.join(root,'results/temp')
    out_path_pred = out_path+'_pred/pred.pkl'
    time = 2019
else:
    feature_list = sys.argv[1].split(',')
    out_path = os.path.join(root,sys.argv[2])
    out_path_pred = out_path+'_pred'
    time = int(sys.argv[3])
    dga = sys.argv[4]

os.makedirs(out_path, exist_ok=True)
os.makedirs(out_path_pred, exist_ok=True)


merged_df = None

if time == 2017:
    time_feature_list = ['uniport_ppi_2017','ppi_2017_dw_80','uniport_exp','uniport_seq','uniport_esm']
elif time == 2019:
    time_feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm','diffusion_2019']

for feature in time_feature_list:
    feature_df = get_feature(root, feature)

    if 'diffusion' in feature:
        pass
    else:
        feature_cols = [col for col in feature_df.columns if col.startswith('feature')]
        if feature_cols:
            scaler = MinMaxScaler()
            feature_df[feature_cols] = scaler.fit_transform(feature_df[feature_cols])

    # Rename columns starting with 'feature'
    feature_df.rename(columns={
        col: f"{feature}_{col}" if col.startswith('feature') else col
        for col in feature_df.columns
    }, inplace=True)

    # Merge iteratively to avoid keeping all DataFrames
    if merged_df is None:
        merged_df = feature_df
    else:
        merged_df = pd.merge(merged_df, feature_df, on='string_id', how='inner')
    del feature_df  # Free memory
name_list = feature_list + ['string_id']

merged_df = merged_df[[col for col in merged_df.columns if any(item in col for item in name_list)]]

if dga == 'disgenet':
    all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv')
elif dga == 'opentarget':
    all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ot_dga_time_uni.csv')
    all_df = all_df[all_df['score']>=0.4]

all_df = all_df[all_df['string_id'].isin(merged_df['string_id'])]
# all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/align_disgent_with_time.csv')

# methods = ['ooc','random_negative','pseudo_labeling','pseudo_labeling_mask']
# methods = ['random_negative','pseudo_labeling','pseudo_labeling_mask','pseudo_labeling_cluster_all_mask']
# methods = ['random_negative','random_negative_bagging','random_pos_negative_bagging']
methods = ['random_negative']

if time_spilt:
    selected_diseases = []
    for disease_id in all_df['disease_id'].unique():
        sub_df = all_df[all_df['disease_id']==disease_id]
        if len(sub_df) < 15:
            continue
        else:
            # print(type(time),type(sub_df['first_pub_year'].max()))
            if sub_df['first_pub_year'].max() > time and sub_df['first_pub_year'].min() <= time and len(sub_df[sub_df['first_pub_year']<time]) >=5:
                selected_diseases.append(disease_id)
else:
    selected_diseases = (
        all_df.groupby('disease_id')
        .filter(lambda x: (len(x) > 15))
        ['disease_id']
        .unique()
        .tolist())
print(feature_list, len(selected_diseases),len(merged_df))
all_results = []

['uniport_ppi_2019', 'ppi_2019_dw_40', 'uniport_bio', 'uniport_seq', 'uniport_esm', 'diffusion_2019'] 48 15686


In [2]:
disease = selected_diseases[0]
if time_spilt:
    df, y = read_data_timecut(disease, all_df, merged_df,time)
else:
    df, y = read_data(disease, all_df, merged_df,time)


In [3]:
result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])

if time_spilt:
    test_idx = df[df['test']==1].index
    train_idx = df[y==1].index.difference(test_idx)
    df.drop(columns='test', inplace=True)

In [ ]:
import numpy as np
import pandas as pd
from sklearn import svm
from rdkit.ML.Scoring.Scoring import CalcBEDROC
# from pseudo_label import select_pseudo_negatives
from sklearn.metrics import roc_auc_score
import os
import pickle
import gseapy as gp
# from concurrent.futures import ProcessPoolExecutor
# import functools
from multiprocessing import Pool
from collections import defaultdict
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.neighbors import NearestNeighbors
from scipy.linalg import eigh
from sklearn.model_selection import StratifiedKFold
from scipy.stats import rankdata
from model_diffusion import compute_kernels, select_gamma_ratio, eval_bagging


train_pos_df = df.loc[train_idx]
test_pos_df = df.loc[test_idx]
neg_num = 5*len(train_pos_df)
neg_df = df[y == 0]
neg_df_add_test_pos = pd.concat([neg_df, test_pos_df])


kernel_dir_path = os.path.join('/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc_norm',str(time))

os.makedirs(kernel_dir_path, exist_ok=True)
kernel_pkl_path = os.path.join(kernel_dir_path,'path_save.pkl')

if os.path.isfile(kernel_pkl_path):
    print('kernels existing')
    with open(kernel_pkl_path, 'rb') as f:
        kernels_all_dict = pickle.load(f)
else:
    kernels_all_dict = dict()

add_feature_list = set(feature_list) - set(kernels_all_dict.keys())
if not add_feature_list:
    pass
else:
    add_feature_list = list(add_feature_list)
####### calculate full kernels for each feature and their logm
    print('calculating kernels...', add_feature_list)
    X_all = []
    
    for feature_name in add_feature_list:
        select_columns = [col for col in df.columns if col.startswith(feature_name)]
        X_all.append(df[select_columns].values)

    args_list = list(zip(X_all, add_feature_list, [kernel_dir_path] * len(X_all), [True] * len(X_all)))
    with Pool(min(len(add_feature_list), os.cpu_count(), 4)) as pool:
        # each tuple (X_feature, feature_id) is unpacked by starmap
        kernel_results = pool.starmap(
            compute_kernels,
            args_list)
    del X_all
    
    for fname, K_s_path_dict in kernel_results:
        kernels_all_dict[fname] = K_s_path_dict
        
    with open(kernel_pkl_path, 'wb') as f:
        pickle.dump(kernels_all_dict, f)
############################## cv get best gamma
args_list = [(neg_df, neg_num, train_pos_df, df, kernels_all_dict[fname], fname)
    for fname in feature_list]

with Pool(processes=len(feature_list)) as pool:
    best_ratios = pool.map(select_gamma_ratio, args_list)

best_ratios_dict = dict()
agg_feature = []
for fname, best_params, best_bedroc, best_auc in best_ratios:
    print(fname, best_params, best_bedroc, best_auc)
    best_ratios_dict[fname] = best_params
    # if best_auc > 0.67 and best_bedroc > 0.5:
    agg_feature.append(fname)
print('collect valid feature: ', agg_feature)
######################### using precalculated kernels to train svm and evaluate, get weights for kernels


kernels existing
uniport_ppi_2019 {'C_num': 9, 'gamma_ratio': 2, 'gamma': '0.5052695526173884'} 0.7183005053156061 0.8637985358168844
ppi_2019_dw_40 {'C_num': 1, 'gamma_ratio': 2, 'gamma': '0.48138034733416246'} 0.7181693220750089 0.8717225406827854
uniport_bio {'C_num': 3, 'gamma_ratio': 8, 'gamma': '0.13568965917980907'} 0.5464803304826745 0.8091873298295317
uniport_seq {'C_num': 27, 'gamma_ratio': 8, 'gamma': '0.05611004556505588'} 0.5277660633941801 0.7882928601583036
uniport_esm {'C_num': 9, 'gamma_ratio': 8, 'gamma': '0.02838808905526428'} 0.4629102431274626 0.7251731307541706
diffusion_2019 {'C_num': 3, 'gamma_ratio': '2', 'gamma': '2'} 0.7177387828981919 0.8828655346239445
collect valid feature:  ['uniport_ppi_2019', 'ppi_2019_dw_40', 'uniport_bio', 'uniport_seq', 'uniport_esm', 'diffusion_2019']


In [5]:
# print('evaluation')

test_neg_df = neg_df
test_df = pd.concat([test_pos_df, test_neg_df])
test_index_loc = df.index.get_indexer(test_df.index)
y_test = np.array([1] * len(test_pos_df) + [0] * len(test_neg_df))


# # test_indices = test_df.index.values
# # enrich_train_genes = train_pos_df.index.values
# # enrich_train_set = enriched_set(enrich_train_genes,time)

num_processes = 15
base_seed = 42
seed_list = [base_seed + i for i in range(num_processes)]

# pathway_overlap_dict = dict()

rank_results_per_feature = dict()
predcition_collection = dict()
predcition_collection['true_label'] = y_test
predcition_collection["test_genes"] = test_df.index
predcition_collection["train_pos_genes"] = train_pos_df.index

In [6]:
feature_list

['uniport_ppi_2019',
 'ppi_2019_dw_40',
 'uniport_bio',
 'uniport_seq',
 'uniport_esm',
 'diffusion_2019']

In [7]:
feature_name = 'diffusion_2019'
gamma = best_ratios_dict[feature_name]['gamma_ratio']
X_path = kernels_all_dict[feature_name][gamma][0]
C_num = best_ratios_dict[feature_name]['C_num']

args_list = [
    (neg_df_add_test_pos, neg_num, train_pos_df, df, X_path, C_num, test_index_loc, seed)
    for seed in seed_list]

In [83]:
len(seed_list)

15

In [79]:
seed = seed_list[15]
neg_df.sample(n=neg_num, replace=True, random_state=seed).equals(neg_df_add_test_pos.sample(n=neg_num, replace=True, random_state=seed))


IndexError: list index out of range

In [80]:
train_neg_df = neg_df.sample(n=neg_num, replace=True, random_state=seed)

train_df = pd.concat([train_pos_df, train_neg_df])
train_index_loc = df.index.get_indexer(train_df.index)
y_train = np.array([1] * len(train_pos_df) + [0] * len(train_neg_df))

if isinstance(X_path, str):
    with open(X_path, 'rb') as f:
        X_all = pickle.load(f)
        X_all = 0.5 * (X_all + X_all.T)
else:
    X_all = X_path

if 'diffusion' in X_path:
    kernel_train_idx = df.loc[df.index[train_index_loc], 'diffusion_2019_feature_0'].values.astype(int)
    kernel_val_idx = df.loc[df.index[test_index_loc], 'diffusion_2019_feature_0'].values.astype(int)

    X_feature_train = X_all[np.ix_(kernel_train_idx, kernel_train_idx)]
    X_feature_test = X_all[np.ix_(kernel_val_idx,kernel_train_idx)]
else:

    X_feature_train = X_all[np.ix_(train_index_loc, train_index_loc)]
    X_feature_test = X_all[np.ix_(test_index_loc,train_index_loc)]

best_svm = svm.SVC(C=C_num, kernel='precomputed')
best_svm.fit(X_feature_train, y_train)
y_scores = best_svm.decision_function(X_feature_test)
overlap = set(train_index_loc)&set(test_index_loc)
mask_loc = [i for i, x in enumerate([overlap]) if x in test_index_loc]

ranked_predict_index, results = eval_bagging(y_scores, y_test)
results

(0.04,
 0.36,
 0.72,
 0.011523687580025609,
 0.016005121638924456,
 0.88,
 0.004694835680751174,
 0.005335040546308152,
 0.977131758661738,
 0.9670053234364067,
 0.9288136350270415,
 0.879121154424291,
 0.8494421188590281,
 0.815647717169604,
 0.7794687257915386,
 0.7463629402756509,
 0.8882739163888176,
 0.1124,
 0.19508084056273595,
 0.4189512221878003,
 0.5416383073817765,
 0.7206565396731331)

In [81]:

train_neg_df2 = neg_df_add_test_pos.sample(n=neg_num, replace=True, random_state=seed)

train_df = pd.concat([train_pos_df, train_neg_df2])
train_index_loc = df.index.get_indexer(train_df.index)
y_train = np.array([1] * len(train_pos_df) + [0] * len(train_neg_df2))

if isinstance(X_path, str):
    with open(X_path, 'rb') as f:
        X_all = pickle.load(f)
        X_all = 0.5 * (X_all + X_all.T)
else:
    X_all = X_path

if 'diffusion' in X_path:
    kernel_train_idx = df.loc[df.index[train_index_loc], 'diffusion_2019_feature_0'].values.astype(int)
    kernel_val_idx = df.loc[df.index[test_index_loc], 'diffusion_2019_feature_0'].values.astype(int)

    X_feature_train = X_all[np.ix_(kernel_train_idx, kernel_train_idx)]
    X_feature_test = X_all[np.ix_(kernel_val_idx,kernel_train_idx)]
else:

    X_feature_train = X_all[np.ix_(train_index_loc, train_index_loc)]
    X_feature_test = X_all[np.ix_(test_index_loc,train_index_loc)]

best_svm = svm.SVC(C=C_num, kernel='precomputed')
best_svm.fit(X_feature_train, y_train)
y_scores = best_svm.decision_function(X_feature_test)
overlap = set(train_index_loc)&set(test_index_loc)
mask_loc = [i for i, x in enumerate([overlap]) if x in test_index_loc]

ranked_predict_index, results = eval_bagging(y_scores, y_test)
results,mask_loc

((0.04,
  0.36,
  0.72,
  0.011523687580025609,
  0.016005121638924456,
  0.88,
  0.004694835680751174,
  0.005335040546308152,
  0.977131758661738,
  0.9670053234364067,
  0.9288136350270415,
  0.879121154424291,
  0.8430384194381005,
  0.815647717169604,
  0.7794687257915386,
  0.7463629402756509,
  0.8713285457809694,
  0.1293,
  0.19437131595354337,
  0.4170412376470745,
  0.5392080305262744,
  0.7168980461999133),
 [])

In [119]:
scores_collect1 = []
results_collect1 = []
for seed in seed_list:
    train_neg_df2 = neg_df.sample(n=neg_num, replace=True, random_state=seed)

    # train_neg_df2 = neg_df_add_test_pos.sample(n=neg_num, replace=True, random_state=seed)

    train_df = pd.concat([train_pos_df, train_neg_df2])
    train_index_loc = df.index.get_indexer(train_df.index)
    y_train = np.array([1] * len(train_pos_df) + [0] * len(train_neg_df2))

    if isinstance(X_path, str):
        with open(X_path, 'rb') as f:
            X_all = pickle.load(f)
            X_all = 0.5 * (X_all + X_all.T)
    else:
        X_all = X_path

    if 'diffusion' in X_path:
        kernel_train_idx = df.loc[df.index[train_index_loc], 'diffusion_2019_feature_0'].values.astype(int)
        kernel_val_idx = df.loc[df.index[test_index_loc], 'diffusion_2019_feature_0'].values.astype(int)

        X_feature_train = X_all[np.ix_(kernel_train_idx, kernel_train_idx)]
        X_feature_test = X_all[np.ix_(kernel_val_idx,kernel_train_idx)]
    else:

        X_feature_train = X_all[np.ix_(train_index_loc, train_index_loc)]
        X_feature_test = X_all[np.ix_(test_index_loc,train_index_loc)]

    best_svm = svm.SVC(C=C_num, kernel='precomputed')
    best_svm.fit(X_feature_train, y_train)
    y_scores = best_svm.decision_function(X_feature_test)
    overlap = set(train_index_loc)&set(test_index_loc)
    mask_loc = [i for i, x in enumerate([overlap]) if x in test_index_loc]
    print(mask_loc)
    scores_collect1.append(y_scores)

    ranked_predict_index, results = eval_bagging(y_scores, y_test)
    results_collect1.append(results)

[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]


In [120]:
np.where(y_test==1)[0]

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24])

In [121]:
mean_arr1 = np.mean(np.array(scores_collect1), axis=0)
mean_arr1[np.where(y_test==1)[0]]

array([-0.64969196, -0.65636406, -0.65172326, -0.65312404, -0.65510917,
       -0.65902951, -0.6609363 , -0.66062292, -0.66021689, -0.66947009,
       -0.65752748, -0.65193204, -0.65651971, -0.65658135, -0.65657383,
       -0.66667272, -0.65367828, -0.65734816, -0.65876923, -0.6514546 ,
       -0.65457225, -0.66609482, -0.66126868, -0.65554224, -0.66119198])

In [141]:
(-mean_arr1).argsort().argsort()[np.where(y_test==1)[0]]

array([    2,   310,    29,    70,   189,   769,  1454,  1310,  1154,
       11252,   478,    37,   331,   336,   335,  6923,   108,   446,
         710,    24,   153,  6197,  1647,   226,  1603])

In [142]:
(-mean_arr2).argsort().argsort()[np.where(y_test==1)[0]]

array([11438,   308, 11449,    69,   187, 11695, 11858,  1305,  1168,
       11252,   479,    36,   331, 11544,   338, 13601,   106, 11601,
       15069,    23,   152, 13443,  1675,   229,  1592])

In [144]:
for single_predict in scores_collect1:
    print((-single_predict).argsort().argsort()[np.where(y_test==1)[0]])

[   10   542    51    77   353  1054  1890  2126  1377 13563   648    60
   352   466   598 12548   166   771  1232    52   232  7878  2596   256
  1820]
[    2   448    57   118   159   782  2028  1636  1350 14260   637    39
   470   446   436 10043   124   527   852    53   162  9186  1828   222
  1905]
[    3   527    25    93   256  1230  1564  2002  1536 14632   686    61
   715   396   464  7931   117   605   758    17   144  7166  2372   371
  4069]
[    7   458    60   134   251   992  1986  1558  1738 15230   763    38
   319   397   396  8303    95   555   862    34   159  6931  2085   231
  1679]
[    8   377    37   124   182   931  1825  3190  1468 13921   527    38
   732   407   510  9363   149   705  1156    27   199 10307  2205   347
  2055]
[    2   315    37    90   242  1310  2609  2054  1960 15263   598    62
   540   699   689  9665   125   747  1168    27   241  7251  2384   499
  2442]
[    6   496    47   105   275  1017  1708  1419  1415 15090   720    69
   

In [143]:
for single_predict in scores_collect2:
    print((-single_predict).argsort().argsort()[np.where(y_test==1)[0]])

[   10   542    51    77   353  1054  1890  2126  1377 13563   648    60
   352   466   598 12548   166   771  1232    52   232  7878  2596   256
  1820]
[    2   448    57   118   159   782  2028  1636  1350 14260   637    39
   470   446   436 10043   124   527   852    53   162  9186  1828   222
  1905]
[    3   492    25    93   254  1265  1620  1994  1579 14636   707    65
   756 15495   481  7935   113   630   779    18   150  7155  2548   406
  4055]
[    7   458 15615   136   254   994  2004  1579  1745 15230   768    41
   320   394   392  8309    94   556   863    34   159  6898  2101   235
  1718]
[    8   377    37   124   182   931  1825  3190  1468 13921   527    38
   732   407   510  9363   149   705  1156    27   199 10307  2205   347
  2055]
[    2   314    36    91   242  1311  2614  2056  1966 15263   598    62
   540   698   691  9663   125   749  1170    27   241 15347  2390   499
  2447]
[    5   508    51   105   281  1031 15431  1415  1492 15088   734    68
   

In [145]:

(-scores_collect2[-6]).argsort().argsort()[np.where(y_test==1)[0]]

array([15406,   490,    62,   121,   334, 15504,  1674,  1225,  1539,
       15205,   637,    54,   445,   404,   580, 10174,   188,   634,
         884,   103,   281,  8560,  2008,   328,  2122])

In [147]:
seed_list[-6]

51

In [152]:
single_neg_ids = neg_df_add_test_pos.sample(n=neg_num, replace=True, random_state=seed_list[-6]).index.tolist()

In [154]:
test_pos_ids = test_pos_df.index.tolist()

In [159]:
train_neg_df2 = neg_df_add_test_pos.sample(n=neg_num, replace=True, random_state=seed_list[-6])
train_df = pd.concat([train_pos_df, train_neg_df2])
train_index_loc = df.index.get_indexer(train_df.index)

In [160]:
set(train_df.index)&set(test_pos_ids)

{'P12830', 'Q99728'}

In [155]:
set(single_neg_ids)&set(test_pos_ids)

{'P12830', 'Q99728'}

In [162]:
len(set(train_index_loc)&set(test_index_loc))

320

In [163]:
overlap = set(train_index_loc)&set(test_index_loc)
mask_loc = [i for i, x in enumerate([overlap]) if x in test_index_loc]
mask_loc

[]

In [166]:
mask_loc = []
for i, x in enumerate(overlap):
    if x in test_index_loc:
        mask_loc.append(i)

In [167]:
mask_loc

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [136]:
scores = np.column_stack((y_test, mean_arr1))  # Stack labels and scores as columns
scores = scores[scores[:, 1].argsort()[::-1]]

In [139]:
scores

array([[ 0.        , -0.64904272],
       [ 0.        , -0.64939767],
       [ 1.        , -0.64969196],
       ...,
       [ 0.        , -0.75590784],
       [ 0.        , -0.75604756],
       [ 0.        , -0.75637316]])

In [123]:
ranked_predict_index1, results1 = eval_bagging(mean_arr1, y_test)
results1[-6:]

(0.9081995383431648,
 0.0925,
 0.2077797652158904,
 0.4526079462201184,
 0.5829230917288402,
 0.7573638625502277)

In [124]:
ranked_predict_index1[np.where(y_test==1)[0]]

array([ 4139,  2385,     0,  4117,  4600,  5295,  4244,  1330,  2382,
        4207,  2411,  1457,  5148,  2533, 13118,  4974,  2024,  2608,
        2950,  1215,  4468,  4536,  5233,  4777,    19])

In [125]:
for a in results_collect1:
    print(a[-6])

0.8706899204924339
0.8782508335470633
0.8783277763529109
0.8846858168761219
0.8705052577583995
0.8699179276737625
0.876755578353424
0.867963580405232
0.8831290074378045
0.8741241343934342
0.8742139010002564
0.8674685816876121
0.8832033854834573
0.8785303924083099
0.8882739163888176


In [191]:
scores_collect2 = []
results_collect2 = []
for seed in seed_list:
    # train_neg_df2 = neg_df.sample(n=neg_num, replace=True, random_state=seed)

    train_neg_df2 = neg_df_add_test_pos.sample(n=neg_num, replace=True, random_state=seed)

    train_df = pd.concat([train_pos_df, train_neg_df2])
    train_index_loc = df.index.get_indexer(train_df.index)
    y_train = np.array([1] * len(train_pos_df) + [0] * len(train_neg_df2))

    if isinstance(X_path, str):
        with open(X_path, 'rb') as f:
            X_all = pickle.load(f)
            X_all = 0.5 * (X_all + X_all.T)
    else:
        X_all = X_path

    if 'diffusion' in X_path:
        kernel_train_idx = df.loc[df.index[train_index_loc], 'diffusion_2019_feature_0'].values.astype(int)
        kernel_val_idx = df.loc[df.index[test_index_loc], 'diffusion_2019_feature_0'].values.astype(int)

        X_feature_train = X_all[np.ix_(kernel_train_idx, kernel_train_idx)]
        X_feature_test = X_all[np.ix_(kernel_val_idx,kernel_train_idx)]
    else:

        X_feature_train = X_all[np.ix_(train_index_loc, train_index_loc)]
        X_feature_test = X_all[np.ix_(test_index_loc,train_index_loc)]

    best_svm = svm.SVC(C=C_num, kernel='precomputed')
    best_svm.fit(X_feature_train, y_train)
    y_scores = best_svm.decision_function(X_feature_test)
    overlap = set(train_index_loc)&set(test_index_loc)
    mask_loc = [np.where(test_index_loc == i)[0][0] for i in overlap]
    scores_collect2.append([y_scores,mask_loc])

    ranked_predict_index, results = eval_bagging(y_scores, y_test)
    results_collect2.append(results)

In [192]:
arrays = np.stack([arr for arr, _ in scores_collect2])          # shape: (n, d)

# Build mask matrix (True = masked / skip)
mask = np.zeros_like(arrays, dtype=bool)
for i, (_, m) in enumerate(scores_collect2):
    mask[i, m] = True

# Invert mask: True where we keep
keep = ~mask

# Compute sum and count efficiently
sum_arr = np.where(keep, arrays, 0).sum(axis=0)
count_arr = keep.sum(axis=0)

final_y_score = np.array(sum_arr / count_arr)

ranked_predict_index3, results3 = eval_bagging(final_y_score, y_test)
results3[-6:]

(0.8731367017183893,
 0.1275,
 0.16768499904977205,
 0.38831921667427577,
 0.5199417437919138,
 0.7119304145715093)

In [171]:
set(y_test)

{0, 1}

In [ ]:
len(overlap),len(train)

321

In [172]:
len(final_y_score),len(y_test)

(15621, 15621)

In [173]:
final_y_score 

array([        nan,         nan,         nan, ..., -0.66811476,
       -0.66546403, -0.68834495])

In [174]:
mask

array([[ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       ...,
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False]])

In [176]:
overlap

{26,
 88,
 137,
 143,
 192,
 236,
 253,
 284,
 403,
 437,
 585,
 683,
 695,
 898,
 938,
 1021,
 1068,
 1072,
 1258,
 1285,
 1370,
 1379,
 1440,
 1463,
 1477,
 1481,
 1570,
 1583,
 1782,
 1783,
 1861,
 1875,
 1989,
 2089,
 2092,
 2228,
 2257,
 2302,
 2321,
 2335,
 2355,
 2358,
 2432,
 2478,
 2521,
 2530,
 2546,
 2558,
 2727,
 2764,
 2784,
 2786,
 2854,
 2957,
 2963,
 2990,
 3105,
 3107,
 3115,
 3119,
 3169,
 3173,
 3199,
 3284,
 3297,
 3299,
 3340,
 3343,
 3353,
 3411,
 3452,
 3466,
 3467,
 3655,
 3656,
 3725,
 3738,
 3815,
 3942,
 3960,
 4029,
 4134,
 4192,
 4316,
 4340,
 4406,
 4435,
 4436,
 4488,
 4503,
 4522,
 4564,
 4566,
 4606,
 4607,
 4644,
 4649,
 4779,
 4810,
 4841,
 4907,
 4939,
 4955,
 4985,
 4987,
 5014,
 5031,
 5041,
 5101,
 5115,
 5164,
 5208,
 5211,
 5240,
 5357,
 5585,
 5615,
 5633,
 5647,
 5685,
 5738,
 5865,
 5881,
 6106,
 6138,
 6291,
 6360,
 6367,
 6377,
 6496,
 6509,
 6583,
 6621,
 6665,
 6686,
 6692,
 6809,
 6906,
 6935,
 6994,
 7012,
 7019,
 7046,
 7074,
 7153,
 7

In [178]:
if 26 in test_index_loc:
    print('1')

1


In [188]:
mask_ids = []
for i in overlap: 
    mask_ids.append(list(np.where(test_index_loc == i)[0])[0])

In [186]:
mask_ids = np.nonzero(np.isin(test_index_loc, np.array(overlap)))[0]

In [190]:
mask_ids = [np.where(test_index_loc == i)[0][0] for i in overlap]

mask_ids

[12229,
 50,
 4120,
 2095,
 2098,
 10229,
 10230,
 14323,
 10238,
 12298,
 14345,
 8221,
 14360,
 112,
 14362,
 4176,
 10281,
 14375,
 10287,
 8260,
 14397,
 14400,
 161,
 167,
 6249,
 12382,
 12398,
 2233,
 216,
 14469,
 10385,
 2261,
 6318,
 4296,
 6325,
 6335,
 8378,
 258,
 14513,
 4320,
 275,
 2304,
 10441,
 8406,
 14544,
 2323,
 12506,
 8427,
 305,
 10466,
 2337,
 10477,
 2357,
 10490,
 2360,
 4383,
 4411,
 4412,
 12567,
 8489,
 6454,
 6466,
 10551,
 14648,
 2434,
 14660,
 4462,
 12619,
 424,
 4477,
 4495,
 12654,
 2479,
 457,
 6540,
 10622,
 14714,
 8593,
 10636,
 12682,
 10649,
 4537,
 4539,
 2521,
 6578,
 2530,
 2545,
 8649,
 4579,
 2557,
 4580,
 6622,
 6643,
 6649,
 4617,
 4622,
 12791,
 12805,
 604,
 14869,
 14878,
 8775,
 8786,
 8803,
 14932,
 6766,
 2726,
 702,
 4750,
 10871,
 714,
 4781,
 2763,
 2782,
 2784,
 12967,
 4812,
 12978,
 15025,
 15029,
 6863,
 8908,
 15048,
 10961,
 8935,
 6892,
 15067,
 13025,
 8947,
 2851,
 4878,
 13039,
 13047,
 15105,
 15108,
 4909,
 15120,


In [127]:
mean_arr2 = np.mean(np.array(scores_collect2), axis=0)
mean_arr2[np.where(y_test==1)[0]]

array([-0.67322665, -0.65639295, -0.67480055, -0.65315862, -0.65515081,
       -0.6818631 , -0.68346128, -0.66064532, -0.6602825 , -0.66947611,
       -0.65757955, -0.65197062, -0.65657919, -0.67939595, -0.65663809,
       -0.68898236, -0.65370752, -0.68039207, -0.70433699, -0.6514945 ,
       -0.65464029, -0.68859115, -0.66134646, -0.65563781, -0.66120723])

In [128]:
ranked_predict_index2, results2 = eval_bagging(mean_arr2, y_test)
results2[-6:]

(0.6649192100538599,
 0.3354,
 0.1294680603610704,
 0.3165617501204507,
 0.40193043649685845,
 0.5157191376635025)

In [129]:
ranked_predict_index2[np.where(y_test==1)[0]]

array([ 4139,  2385,  4117,  4600,  5295,  4244,  1330,  2382,  4207,
        5148,  2411,  1457,  2533, 13118,  4974,  2024,  2608,  2950,
        1215,  4468,  4536,  5233,  4777,    19,  2292])

In [130]:
ranked_predict_index1[np.where(y_test==1)[0]]

array([ 4139,  2385,     0,  4117,  4600,  5295,  4244,  1330,  2382,
        4207,  2411,  1457,  5148,  2533, 13118,  4974,  2024,  2608,
        2950,  1215,  4468,  4536,  5233,  4777,    19])

In [131]:
mean_arr1[2],mean_arr2[2]

(-0.651723256096351, -0.674800551065573)

In [133]:
np.max(mean_arr1)

-0.649042718874865

In [ ]:
y_scores, mask_loc = neg_bagging(args_list[0])

In [ ]:
# Step 2: Use Pool to parallelize
with Pool(processes=num_processes) as pool:
    bagging_y_scores_with_mask = pool.map(neg_bagging, args_list)

arrays = np.stack([arr for arr, _ in bagging_y_scores_with_mask])          # shape: (n, d)

# Build mask matrix (True = masked / skip)
mask = np.zeros_like(arrays, dtype=bool)
for i, (_, m) in enumerate(bagging_y_scores_with_mask):
    mask[i, m] = True

# Invert mask: True where we keep
keep = ~mask

# Compute sum and count efficiently
sum_arr = np.where(keep, arrays, 0).sum(axis=0)
count_arr = keep.sum(axis=0)

final_y_score = np.array(sum_arr / count_arr)

# final_y_score = np.mean(bagging_y_scores, axis=0)

# pathway_overlap_dict[feature_name] = jac_sm

ranked_predict_index, results = eval_bagging(final_y_score, y_test)
# Add results to the result dataframe
result_df.loc[len(result_df.index)] = ["random_negative",fold,feature_name+'-0-0-0', *results]
rank_results_per_feature[feature_name] = rankdata(final_y_score, method='average')
predcition_collection[feature_name] = final_y_score